- 컴퓨너에서 난수

    1. 수학 공식으로 계산되어 시작점(seed)이 같으면 결과도 100% 같습니다.

    2. 라이브러리마다 사용하는 난수 생성 공식이 다르기 때문에 같은 seed라도 결과는 다릅니다.

    - Python, Java, PyTorch, TensorFlow 등 각자 다른 난수 생성기(Random Number Generator, RNG) 사용 합니다.
    - 같은 라이브러리 안에서는 seed만 같으면 항상 재현이 가능합니다.

    3. 파이토치는 CPU용 난수 생성기와 GPU요 난수 생성기를 따로 관리합니다.

        | 구분 | 의미 | seed 설정 |
        | :--- | :--- | :--- |
        | CPU 난수 | CPU에서 동작하는 RNG가 만든 값 | `torch.manual_seed()` |
        | GPU 난수 | GPU(CUDA)에서 동작하는 RNG가 만든 값 | `torch.cuda.manual_seed()` |

- 결정론 (Deterministic)과 비결정론(Non-deterministic)
    - 결정론(deterministic)과 비결정론(non-deterministic)은 같은 입력에 대해 계산 과정과 결가가 어떻게 나타나는가를 설명하는 개념입니다.

    1. 결정론 (Deterministic)
        - 같은 입력이 주어지면 계산 과정이 항상 동일하게 진행되며, 그 결과도 언제나 같습니다.

        - 실행 시점이나 환경이 달려도 결과가 변하지 않기 때문에 예측 가능하며, 실험의 재현성과 디버깅에 유리합니다.

        - 같은 입력 → 같은 계산 과정 → 항상 같은 결과

    2. 비결정론 (Non-deterministic)

        - 같은 입력이 주어지더라도 실행할 때마다 계산 과정이나 결과가 달라질 수 있습니다.

        - 주로 병렬 처리 환경에서 발생하며, 연산 순서가 실행마다 달라질 수 있기 때문입니다.

    - 예를 들어 CuDNN(엔비디아 딥러닝 라이브러리)에서는 수학적으로 동일한 합성곱 연산이라도 여러 내부 구현 알고리즘이 존재하며, GPU 구조와 입력 조건에 따라 가장 빠른 계산 방법이 선택됩니다. 이로 인해 실행마다 다른 알고리즘이 사용될 수 있고, 그 결과 비결정론적 특성이 나타날 수 있습니다.
    - 딥러닝은 GPU 병렬 연산 등으로 인해 실행할 때마다 결과가 조금씩 달라질 수 있습니다. 따라서 실험 결과는 한 번의 실행 결과가 아니라 여러 번 실행한 뒤 평균과 표준편차를 이용해 분석합니다.

In [1]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 실행마다 동일한 결과를 얻기 위해 파이토치에 랜덤 시드를 지정하고 GPU 연산을 결정적으로 만듭니다.
import torch

torch.manual_seed(42) # CPU에서 발생하는 난수(랜덤 값)를 42번 시드로 고정합니다.

# CUDA(쿠다)는 CUDA(Compute Unified Device Architecture)는 그래픽 카드 제조사인 NVIDIA(엔비디아) 가 만든 병렬 컴퓨팅 플랫폼입니다.
if torch.cuda.is_available():                   # 현재 컴퓨터에 NVIDIA GPU가 있는지, 그리고 CUDA 드라이버가 잘 설치되어 있는지 확인합니다.
    torch.cuda.manual_seed(42)                  # GPU(CUDA) 내부에서 사용하는 난수도 42번으로 고정합니다.
    torch.backends.cudnn.deterministic = True   # cuDNN은 CUDA를 딥러닝에 최적화한 라이브러리입니다.
                                                # 기본적으로 cuDNN은 가장 빠른 계산 경로를 찾기 위해 내부 알고리즘을 매번 조금씩 바꾸는데,
                                                # 옵션을 True로 하면 무조건 똑같은 경오로 계산하게 하여 결과 오차를 없앱니다.

In [3]:
import torch    # PyTorch 핵심 라이브러리

# otrchvision : 이미지 처리와 컴퓨터 비전 작업을 위한 PyTorch 부속 라이브러리입니다.
# datasets : 다양한 공개 데이터셋을 쉽게 다운로드하고 사용할 수 있도록 제공하는 모듈입니다.
# FashionMNIST : 의류 이미지 데이터셋을 불러오기 위한 클래스입니다.
from torchvision.datasets import FashionMNIST

In [4]:
# root='./data_torch' : 다운로드 위치 (현재 디렉터리 기준 ./data_torch/FashionMNIST/ 폴더에 저장됨)
# train=True : 훈련용 데이터셋을 불러옵니다.
# download=True : 원격에 저장된 데이터를 다운로드하여 로컬에 저장, 이미 로컬에 데이터가 있으면 다시 다운로드 하지 않습니다.
fm_train = FashionMNIST(root='./data_torch', train=True, download=True)

# train=False : 테스트 데이터 (10,000장)를 불러옵니다.
fm_test = FashionMNIST(root='./data_torch', train=False, download=True)

In [5]:
print(fm_train.data.shape, fm_test.data.shape)
# torch.Size([60000, 28, 28]) : 6만개 학습용 이미지 개수 / 28 : 이미지 높이 / 28 : 이미지 너비
# torch.Size([10000, 28, 28]) : 만개 테스트 이미지 개수 / 28 : 이미지 높이 / 28 : 이미지 너비

torch.Size([60000, 28, 28]) torch.Size([10000, 28, 28])


In [6]:
print(fm_train.targets.shape, fm_test.targets.shape)
# 타겟이 1차원 이므로 원-핫 인코딩이 아니라 정숫값이라는 것을 알 수 있습니다.

torch.Size([60000]) torch.Size([10000])


In [7]:
train_input=  fm_train.data
train_target = fm_train.targets

In [8]:
# 정규화
train_scaled = train_input / 255.0

In [9]:
from sklearn.model_selection import train_test_split

train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

In [10]:
# train_scaled 는 6만개의 80% -> 48000개
# val_scaled 는 6만개의 20% -> 12000개
print(train_scaled.shape, val_scaled.shape)

torch.Size([48000, 28, 28]) torch.Size([12000, 28, 28])


 ---

In [11]:
import torch.nn as nn

# 784개의 입력과 100개읭 뉴런을 가진 은닉층, 10개의 뉴런을 갖는 출력층을 구성해 봅시다.

# 파이토치는 모델의 입력 크기를 사전에 지정할 필요가 없습니다.

# 케라스의 Dense층과 동일한 역할을 하는 것이 파이토치의 Linear() 입니다.
# Linear(입력크기, 출력 크기-뉴런 개수)를 사용합니다.

# ReLU() : 활성화 함수를 별도의 층으로 추가해야 합니다.
# 출력층 다음에 활성화 함수가 없습니다.
model = nn.Sequential(
    nn.Flatten(),           # 2차원 이미지(28 * 28)를 1차원(784)으로 길게 펼쳐주는 역할을 합니다.
    nn.Linear(784, 100),    # 784개의 입력을 받아 100개의 뉴런으로 연결합니다. Linear(입력 크기, 출력 크기-뉴련 개수)를 사용합니다.
    nn.ReLU(),              # 활성화 함수입니다.
    nn.Linear(100, 10)      # 최종 10개 클래스(티셔츠, 바지 등)로 분류하는 출력층입니다.
)

In [12]:
# torchinfo 패키지를 설치해서 케라스의 summary() 와 비슷한 결과를 얻을 수 있습니다.
%pip install torchinfo

Note: you may need to restart the kernel to use updated packages.


In [13]:
from torchinfo import summary

In [ ]:
# 32는 배치 사이즈를 의미합니다.
summary(model, input_size=(32, 28, 28))
# 78,500 = 가중치(weight) + 편향(bias) = 784 * 100 + 100
#  1,010 = 가중치(weight) + 편향(bias) = 100 * 10 + 10

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [32, 10]                  --
├─Flatten: 1-1                           [32, 784]                 --
├─Linear: 1-2                            [32, 100]                 78,500
├─ReLU: 1-3                              [32, 100]                 --
├─Linear: 1-4                            [32, 10]                  1,010
Total params: 79,510
Trainable params: 79,510
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 2.54
Input size (MB): 0.10
Forward/backward pass size (MB): 0.03
Params size (MB): 0.32
Estimated Total Size (MB): 0.45

Param # (78,500 / 1,010): 모델이 학습해야 할 가중치(Weight)와 편향(Bias)의 개수입니다.

Total mult-adds (2.54 MB): 데이터를 처리할 때 발생하는 곱셈과 덧셈 연산량입니다.

Input size (0.10 MB): 들어오는 데이터(이미지 32장)의 크기입니다.

Params size (0.32 MB): 모델의 가중치들(79,510개)이 차지하는 용량입니다.

Estimated Total Size (0.45 MB): 입력 데이터, 중간 활성값(activation), 그리고 모델의 파라미터가 차지하는 메모리를 합한 전체 메모리 사용량의 추정값입니다.

---

In [15]:
# 케라스의 경우 GPU가 감지 되면 자동으로 모델을 GPU에서 훈련합니다.
# 반면 PyTorch는 모델과 데이터를 명시적으로 GPU로 이동해야 합니다.
# torch.cuda.is_available() 함수는 CUDA를 사용할 수 있는 GPU가 있는지 확인합니다.
# NVIDIA GPU와 CUDA 드라이버가 정상적으로 설치되어 있으면 True를 반환하고,
# 그렇지 않으면 False를 반환합니다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=10, bias=True)
)

In [17]:
# 손실 함수와 옵티마이저
import torch.optim as optim

# CrossEntropyLoss는 소프트맥스 함수 계산과 크로스 엔트로피 계산을 하나의 연산으로 합쳐 효율적으로 계산할 수 있도록 설계되어 있습니다.
# 다중 분류 문제를 다룰 때 파이토치 모델의 마지막에 소프트맥스 함수를 추가할 필요가 없습니다.
criterion = nn.CrossEntropyLoss()

# model.parameters() : 학습 대상인 모든 가중치 + bias
optimizer = optim.Adam(model.parameters())

In [18]:
for params in model.parameters():
    print(params.shape)

# torch.Size([100, 784]) : 첫 번째 Linear 층의 가중치로 100은 은닉층 뉴런의 개수
# torch.Size([100])      : 첫 번째 Linear 층의 절편, 100은 은닉층 뉴런 100개 각각의 bias
# torch.Size([10, 100])  : 두 번째 Linear 층의 가중치로 10은 출력층 뉴런의 개수
# torch.Size([10])       : 두 번째 Linear 층의 절편

torch.Size([100, 784])
torch.Size([100])
torch.Size([10, 100])
torch.Size([10])


In [19]:
print(len(train_scaled))
print(len(train_scaled)/32)

48000
1500.0


In [21]:
# 총 학습 반복 횟수(Epoch)
epochs = 5

# 전체 훈련 데이터를 배치 크기(32)로 나누어 몇 번 반복할지 계산
batches = int(len(train_scaled)/32) # 1500

for epoch in range(epochs):

    model.train()   # 모델을 학습 모드로 설정

    train_loss = 0  # 한 에포크 동안 누적할 손실값 초기화

    # 미니배치 단위로 학습
    for i in range(batches):

        # 현재 배치에 해당하는 입력 데이터를 가져옴
        # GPU를 사용할 경우 device(GPU)로 이동
        inputs = train_scaled[i*32:(i+1)*32].to(device)

        # 현재 배치에 해당하는 정답(label) 데이터를 가져옴
        targets = train_target[i*32:(i+1)*32].to(device)

        # 이전 배치에서 계산된 gradient를 초기화
        # PyTorch는 gradient를 누적(accumulate)하기 때문에 매 배치마다 0으로 초기화해야 함
        optimizer.zero_grad()

        # 순전파 (Forward Propagation)
        # 입력 데이터를 모델에 넣어 예측값(outputs)을 계산
        outputs = model(inputs)

        # 예측값(outputs)과 실제값(targets)을 비교하여 손실(loss)을 계산
        loss = criterion(outputs, targets)

        # 역전파 (Backpropagation)
        # 손실을 기준으로 각 가중치에 대한 gradient 계산
        loss.backward()

        # 옵티마이저가 gradient를 이용해 가중치를 업데이트
        optimizer.step()

        # 현재 배치의 손실 값을 누적
        # loss.item()은 tensor 값을 Python 숫자로 변환
        train_loss += loss.item()

    # 한 에포크 동안의 평균 손실 출력
    print(f"에포크:{epoch+1}, 손실:{train_loss/batches:.4f}")

에포크:1, 손실:0.2951
에포크:2, 손실:0.2803
에포크:3, 손실:0.2685
에포크:4, 손실:0.2581
에포크:5, 손실:0.2479


 ---
정확도를 계산하기 위한 평가 단계

In [24]:
# 모델을 평가 모드로 전환합니다.
model.eval()

# PyTorch는 기본적으로 역전파를 위해 모든 연산 과정을 기록하지만
# 평가 단계에서는 가중치를 수정하지 않으므로 이 기록 과정이 불필요합니다.
# torch.no_grad()를 사용하면 중간 연산값 저장에 필요한 메모리를 절약할 수 있습니다.
with torch.no_grad():
    # 검증 데이터를 모델이 있는 장치(cpu 또는 gpu)로 이동합니다.
    val_scaled = val_scaled.to(device)

    # 정답(label) 데이터도 같은 장치(cpu 또는 gpu)로 이동합니다.
    val_target = val_target.to(device)

    # 모델에 입력 데이터를 넣어 예측 결과를 계산합니다. (순전파 수행)
    # outputs의 형태는 batch_size, 10) 즉 각 샘플에 대해 10개 클래스의 점수(logits)
    outputs = model(val_scaled)

    # outputs의 형태는 (데이터 개수, 클래스 개수) 입니다.
    # 예: (10000, 10) → 10000개 데이터, 각 데이터마다 10개 클래스 점수
    # argmax는 가장 큰 값의 인덱스를 반환합니다.
    # dim=1 은 "클래스 축"에서 가장 큰 값을 선택하라는 의미입니다.
    predicts = torch.argmax(outputs, 1)

    # 예측값(predicss)과 실제 정답(val_target)이 같은지 비교합니다.
    # True는 1, False는 0으로 계산됩니다.
    # sum()으로 맞은 개수를 구합니다.
    # item()은 tensor 값을 Python 숫자로 변환합니다.
    corrects = (predicts == val_target).sum().item()

    print(corrects)

# 정확도 = 맞은 개수 / 전체 데이터 개수
accuracy = corrects / len(val_target)

# 소수점 4자리까지 출력
print(f"검증 정확도: {accuracy:.4f}")

10534
검증 정확도: 0.8778


In [22]:
import torch

# 원소가 1개인 텐서 생성
loss = torch.tensor(0.1234)
print(type(loss))

# 텐서에서 순수한 파이썬 숫자(float)만 추출
value = loss.item()

print(value)        # 출력: 0.1234
print(type(value))  # 출력: <class 'float'>

<class 'torch.Tensor'>
0.1234000027179718
<class 'float'>


In [23]:
# 여러 개의 값이 담긴 텐서 (예: 모델의 예측값들)
predicts = torch.tensor([3, 0, 9])

# 파이썬 리스트로 변환
pred_list = predicts.tolist()

print(pred_list)        # 출력: [3, 0, 9]
print(type(pred_list))  # 출력: <class 'list'>

[3, 0, 9]
<class 'list'>
